# Adaptation Pipeline Runner

Clean runner notebook sourced from `AdaptationPipeline_nblazek_setcount_exec.py`.

Use the setup cells first, then run exactly one dataset cell at a time.
Each runner cell creates a fresh agent and prints a short summary with result paths.


In [ ]:
import os
import json
from pathlib import Path


## Bootstrap

Loads everything from the current exported script up to the invoke section.
This keeps the notebook aligned with the `py.py` source of truth.


In [ ]:
# Locate agents directory and load the pipeline definitions from the exported script.
ROOT = Path.cwd()
AGENTS_DIR = ROOT if (ROOT / "AdaptationPipeline_nblazek_setcount_exec.py").exists() else ROOT / "agents"
SCRIPT_PATH = AGENTS_DIR / "AdaptationPipeline_nblazek_setcount_exec.py"

if not SCRIPT_PATH.exists():
    raise FileNotFoundError(f"Could not find {SCRIPT_PATH}")

os.chdir(AGENTS_DIR)
source = SCRIPT_PATH.read_text(encoding="utf-8")
split_marker = "# ## Invoke Pipeline"
if split_marker not in source:
    raise RuntimeError(f"Split marker not found: {split_marker}")

bootstrap = source.split(split_marker, 1)[0]
ns = {"__name__": "__main__"}
exec(bootstrap, ns)

INPUT_DIR = ns.get("INPUT_DIR", "input/")
OUTPUT_DIR = ns.get("OUTPUT_DIR", "output/")
print(f"Loaded bootstrap from: {SCRIPT_PATH}")
print(f"AGENTS_DIR: {AGENTS_DIR}")


## Helpers

These helpers create a fresh agent per run and print a compact summary.


In [ ]:
ChatOpenAI = ns["ChatOpenAI"]
ProfileDatasetTool = ns["ProfileDatasetTool"]
SearchDocumentationTool = ns["SearchDocumentationTool"]
SimpleModelAgent = ns["SimpleModelAgent"]


def make_agent(model: str = "gpt-5.2", max_retries: int = 5):
    llm = ChatOpenAI(
        model=model,
        temperature=0,
        max_tokens=None,
        timeout=ns.get("OPENAI_REQUEST_TIMEOUT", 600),
        max_retries=max_retries,
    )
    profile_tool = ProfileDatasetTool()
    search_tool = SearchDocumentationTool()
    return SimpleModelAgent(
        llm,
        tools={profile_tool.name: profile_tool, search_tool.name: search_tool},
    )


def run_case(
    case_name: str,
    datasets,
    entity_matching_testsets,
    fusion_testset,
    validation_fusion_testset=None,
    matcher_mode: str = "rulebased",
    skip_blocking: bool = False,
    skip_matching: bool = False,
    recursion_limit: int = 200,
):
    ns["SKIP_BLOCKING_TESTER"] = skip_blocking
    ns["SKIP_MATCHING_TESTER"] = skip_matching

    agent = make_agent()

    payload = {
        "datasets": datasets,
        "original_datasets": list(datasets),
        "normalized_datasets": [],
        "entity_matching_testsets": entity_matching_testsets,
        "fusion_testset": fusion_testset,
        "matcher_mode": matcher_mode,
        "evaluation_attempts": 0,
        "normalization_attempts": 0,
        "normalization_execution_result": "",
        "normalization_rework_required": False,
        "normalization_rework_reasons": [],
        "normalization_directives": {},
        "investigator_decision": "",
    }
    if validation_fusion_testset:
        payload["validation_fusion_testset"] = validation_fusion_testset

    print(f"[*] Starting {case_name} run...")
    result = agent.graph.invoke(payload, config={"recursion_limit": recursion_limit})
    print(f"[*] {case_name} run complete")

    summary = {
        "case": case_name,
        "validation_overall_accuracy": (result.get("evaluation_metrics") or {}).get("overall_accuracy"),
        "validation_macro_accuracy": (result.get("evaluation_metrics") or {}).get("macro_accuracy"),
        "sealed_test_overall_accuracy": (result.get("sealed_test_metrics_final") or {}).get("overall_accuracy"),
        "sealed_test_macro_accuracy": (result.get("sealed_test_metrics_final") or {}).get("macro_accuracy"),
        "normalization_attempts": result.get("normalization_attempts"),
        "evaluation_attempts": result.get("evaluation_attempts"),
        "investigator_decision": result.get("investigator_decision"),
        "run_id": result.get("run_id"),
        "run_output_root": result.get("run_output_root"),
        "run_audit_path": result.get("run_audit_path"),
        "run_report_path": result.get("run_report_path"),
    }
    print(json.dumps(summary, indent=2, default=str))
    return result, summary


## Companies

Notebook-exact companies block with validation and sealed final test.


In [ ]:
companies_datasets = [
    INPUT_DIR + "datasets/companies/forbes.csv",
    INPUT_DIR + "datasets/companies/dbpedia.xml",
    INPUT_DIR + "datasets/companies/fullcontact.xml",
]
companies_entity_matching_testsets = {
    ("forbes", "dbpedia"): INPUT_DIR + "datasets/companies/testsets/forbes_dbpedia_art.csv",
    ("forbes", "fullcontact"): INPUT_DIR + "datasets/companies/testsets/forbes_fullcontact_art.csv",
    ("fullcontact", "dbpedia"): INPUT_DIR + "datasets/companies/testsets/fullcontact_dbpedia_art.csv",
}
companies_fusion_testset = INPUT_DIR + "datasets/companies/testsets/test_set.xml"
companies_validation_fusion_testset = INPUT_DIR + "datasets/companies/testsets/validation_set.xml"

RESULT_COMPANIES, SUMMARY_COMPANIES = run_case(
    case_name="companies",
    datasets=companies_datasets,
    entity_matching_testsets=companies_entity_matching_testsets,
    fusion_testset=companies_fusion_testset,
    validation_fusion_testset=companies_validation_fusion_testset,
)


## Games

Uses the `_2_` entity matching sets and the fusion/validation XML files.


In [ ]:
games_datasets = [
    INPUT_DIR + "datasets/games/dbpedia.xml",
    INPUT_DIR + "datasets/games/metacritic.xml",
    INPUT_DIR + "datasets/games/sales.xml",
]
games_entity_matching_testsets = {
    ("dbpedia", "sales"): INPUT_DIR + "datasets/games/testsets/dbpedia_2_sales_test.csv",
    ("metacritic", "dbpedia"): INPUT_DIR + "datasets/games/testsets/metacritic_2_dbpedia_test.csv",
    ("metacritic", "sales"): INPUT_DIR + "datasets/games/testsets/metacritic_2_sales_test.csv",
}
games_fusion_testset = INPUT_DIR + "datasets/games/testsets/test_set_fusion.xml"
games_validation_fusion_testset = INPUT_DIR + "datasets/games/testsets/validation_set_fusion.xml"

RESULT_GAMES, SUMMARY_GAMES = run_case(
    case_name="games",
    datasets=games_datasets,
    entity_matching_testsets=games_entity_matching_testsets,
    fusion_testset=games_fusion_testset,
    validation_fusion_testset=games_validation_fusion_testset,
)


## Music

Uses the existing music blocking gold standards plus validation and sealed final test XMLs.


In [ ]:
music_datasets = [
    INPUT_DIR + "datasets/music/discogs.xml",
    INPUT_DIR + "datasets/music/lastfm.xml",
    INPUT_DIR + "datasets/music/musicbrainz.xml",
]
music_entity_matching_testsets = {
    ("discogs", "lastfm"): INPUT_DIR + "datasets/music/testsets/discogs_lastfm_goldstandard_blocking.csv",
    ("discogs", "musicbrainz"): INPUT_DIR + "datasets/music/testsets/discogs_musicbrainz_goldstandard_blocking.csv",
    ("musicbrainz", "lastfm"): INPUT_DIR + "datasets/music/testsets/musicbrainz_lastfm_goldstandard_blocking.csv",
}
music_fusion_testset = INPUT_DIR + "datasets/music/testsets/test_set.xml"
music_validation_fusion_testset = INPUT_DIR + "datasets/music/testsets/validation_set.xml"

RESULT_MUSIC, SUMMARY_MUSIC = run_case(
    case_name="music",
    datasets=music_datasets,
    entity_matching_testsets=music_entity_matching_testsets,
    fusion_testset=music_fusion_testset,
    validation_fusion_testset=music_validation_fusion_testset,
)


## Restaurants

Uses the available fusion test set only. No validation split is configured here.
If you later want a validation/test split, create it once and wire both paths into the runner.


In [ ]:
restaurant_datasets = [
    INPUT_DIR + "datasets/restaurant/kaggle_small.parquet",
    INPUT_DIR + "datasets/restaurant/uber_eats_small.parquet",
    INPUT_DIR + "datasets/restaurant/yelp_small.parquet",
]
restaurant_entity_matching_testsets = {
    ("kaggle_small", "uber_eats_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_uber_eats_goldstandard_blocking_small.csv",
    ("kaggle_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/kaggle_yelp_goldstandard_blocking_small.csv",
    ("uber_eats_small", "yelp_small"): INPUT_DIR + "datasets/restaurant/testsets/uber_eats_yelp_goldstandard_blocking_small.csv",
}
restaurant_fusion_testset = INPUT_DIR + "datasets/restaurant/testsets/Restaurant_Fusion_Test_Set.csv"

RESULT_RESTAURANT, SUMMARY_RESTAURANT = run_case(
    case_name="restaurant",
    datasets=restaurant_datasets,
    entity_matching_testsets=restaurant_entity_matching_testsets,
    fusion_testset=restaurant_fusion_testset,
    validation_fusion_testset=None,
)


## Books

Uses the small parquet datasets and the golden fused books file as the fusion test set.


In [ ]:
books_datasets = [
    INPUT_DIR + "datasets/books/amazon_small.parquet",
    INPUT_DIR + "datasets/books/goodreads_small.parquet",
    INPUT_DIR + "datasets/books/metabooks_small.parquet",
]
books_entity_matching_testsets = {
    ("goodreads_small", "amazon_small"): INPUT_DIR + "datasets/books/testsets/goodreads_2_amazon.csv",
    ("metabooks_small", "amazon_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_amazon.csv",
    ("metabooks_small", "goodreads_small"): INPUT_DIR + "datasets/books/testsets/metabooks_2_goodreads.csv",
}
books_fusion_testset = INPUT_DIR + "datasets/books/testsets/golden_fused_books.csv"

RESULT_BOOKS, SUMMARY_BOOKS = run_case(
    case_name="books",
    datasets=books_datasets,
    entity_matching_testsets=books_entity_matching_testsets,
    fusion_testset=books_fusion_testset,
    validation_fusion_testset=None,
)
